In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
import re
from transformers import LlamaTokenizer, AutoModelForCausalLM, pipeline

model_name = "slovak-nlp/mistral-sk-7b"

print(f"Loading {model_name}...")

tokenizer = LlamaTokenizer.from_pretrained(
    model_name,
    clean_up_tokenization_spaces=True,
    legacy=False,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)
model.eval()

# Resolve the device from model parameters (device_map="auto" can split across devices)
device = next(model.parameters()).device

# Mistral does not define a pad token; use EOS as a safe fallback
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Pipeline approach
# text_gen_pipeline = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )

# print("\n--- Text Generation Example (Pipeline Way) ---")

# prompt_pipeline_1 = "Polícia objasnila prípad, v ktorom"
# result_pipeline_1 = text_gen_pipeline(prompt_pipeline_1, max_new_tokens=200, do_sample=True, temperature=0.7)
# print(f"\nPrompt: '{prompt_pipeline_1}'")
# print(f"Generated: {result_pipeline_1[0]['generated_text']}")

# prompt_pipeline_2 = "Podľa najnovších správ slovenská vláda plánuje"
# result_pipeline_2 = text_gen_pipeline(prompt_pipeline_2, max_new_tokens=200, do_sample=True, temperature=0.7)
# print(f"\nPrompt: '{prompt_pipeline_2}'")
# print(f"Generated: {result_pipeline_2[0]['generated_text']}")

print("\n--- Manual Text Generation Example ---")

def fix_sentencepiece_output(text):
    # Replace the SentencePiece whitespace marker with a real space
    text = text.replace("▁", " ")
    # Collapse spaces that got split around punctuation
    text = re.sub(r" ([.,;:!?\"\')\]\}])", r"\1", text)
    text = re.sub(r"([\(\"'\[\{]) ", r"\1", text)
    # Remove spaces between digits (e.g. "1 6" -> "16")
    text = re.sub(r"(?<=\d) (?=\d)", "", text)
    # Remove spaces around colons/dots between digits (e.g. "17 : 00" -> "17:00")
    text = re.sub(r"(\d) ([:.]) (\d)", r"\1\2\3", text)
    # Collapse any remaining multiple spaces
    text = re.sub(r" {2,}", " ", text)
    text = text.strip()
    return text

def generate_text_manual(
    prompt,
    model,
    tokenizer,
    device,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.90,
    repetition_penalty=1.1,
    stop_strings=None,
):
    # 1. Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        max_length=1024,
    ).to(device)

    # 2. Build generate() kwargs — stop_strings requires tokenizer to be passed in too
    generate_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if stop_strings is not None:
        generate_kwargs["stop_strings"] = stop_strings
        generate_kwargs["tokenizer"] = tokenizer

    # 3. Generate tokens
    with torch.no_grad():
        output_ids = model.generate(**generate_kwargs)

    # 4. Decode prompt and full output through the same full-sequence path,
    #    then fix SentencePiece artifacts in both before splitting
    prompt_decoded = fix_sentencepiece_output(
        tokenizer.decode(
            inputs["input_ids"][0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
    )
    full_text = fix_sentencepiece_output(
        tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
    )

    # 5. Strip using the decoded prompt length; lstrip() cleans the split boundary
    generated_text = full_text[len(prompt_decoded):].lstrip()

    return {
        "prompt": prompt_decoded,
        "generated": generated_text,
        "full_text": full_text,
    }

# Example 1: News-article style (lukasjanek/slovaksum domain)
prompt_1 = "Polícia objasnila prípad, v ktorom"
result_1 = generate_text_manual(prompt_1, model, tokenizer, device, max_new_tokens=200, stop_strings=["\n"])
print(f"\nPrompt: '{result_1['prompt']}'")
print(f"Generated continuation: '{result_1['generated']}'")
print(f"Full text: '{result_1['full_text']}'")

# Example 2: Current-affairs/government prompt
prompt_2 = "Podľa najnovších správ slovenská vláda plánuje"
result_2 = generate_text_manual(prompt_2, model, tokenizer, device, max_new_tokens=200, stop_strings=["\n"])
print(f"\nPrompt: '{result_2['prompt']}'")
print(f"Generated continuation: '{result_2['generated']}'")
print(f"Full text: '{result_2['full_text']}'")

Loading slovak-nlp/mistral-sk-7b...


tokenizer_config.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/673 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


--- Manual Text Generation Example ---

Prompt: 'Polícia objasnila prípad, v ktorom'
Generated continuation: 'sa pred niekoľkými dňami našiel mŕtvy 56-ročný muž z obce Kamenica nad Cirochou.'
Full text: 'Polícia objasnila prípad, v ktorom sa pred niekoľkými dňami našiel mŕtvy 56-ročný muž z obce Kamenica nad Cirochou.'

Prompt: 'Podľa najnovších správ slovenská vláda plánuje'
Generated continuation: 'vybudovať v hlavnom meste novú nemocnicu. Štátna investícia by mala stáť 365 miliónov eur, pričom na jej výstavbu sa má použiť aj peniaze z plánu obnovy a odolnosti.'
Full text: 'Podľa najnovších správ slovenská vláda plánuje vybudovať v hlavnom meste novú nemocnicu. Štátna investícia by mala stáť 365 miliónov eur, pričom na jej výstavbu sa má použiť aj peniaze z plánu obnovy a odolnosti.'


In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
